# ERP RAG — External LLM & Embedding Server (Kaggle)

This notebook starts the two backend servers that the ERP Agentic RAG system
connects to when deployed on Kaggle:

| Server | Port | Purpose |
|--------|------|---------|
| vLLM OpenAI-compatible LLM server | 8000 | Answer generation (replaces Gemini API) |
| Embedding server (uvicorn) | 8001 | Chunk embedding for vector search |

**When to use this notebook:**
- You are running on a Kaggle GPU instance and want a free, self-hosted LLM.
- You want to test the full RAG pipeline without an external API key.
- You are benchmarking vLLM latency against the Gemini free-tier client.

Run the cells top-to-bottom. GPU-only cells are marked and can be skipped
if you are on a CPU-only environment.

## Section 1 — Install Dependencies

Install `vllm` and its dependencies. This step requires a GPU and several
minutes to complete. Skip this cell if vLLM is already installed or if you
are using the Gemini API instead.

In [ ]:
# GPU-only: install vLLM (takes 3-5 minutes on Kaggle)
import subprocess
result = subprocess.run(
    ["pip", "install", "vllm", "--quiet"],
    capture_output=True,
    text=True,
)
print(result.stdout or "vllm install complete")
if result.returncode != 0:
    print("STDERR:", result.stderr)

## Section 2 — Start the vLLM OpenAI-Compatible Server

This launches a background process that exposes an OpenAI-compatible
`/v1/chat/completions` endpoint on port **8000**.

The `vLLMLLMClient` in `src/infrastructure/generation/vllm_llm_client.py`
connects to this endpoint. Set `VLLM_MODEL` to any HuggingFace model ID
that fits in your Kaggle GPU memory (e.g. `mistralai/Mistral-7B-Instruct-v0.2`).

In [ ]:
# GPU-only: start vLLM server in the background
import subprocess
import time

VLLM_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"  # change to your preferred model
VLLM_PORT = 8000

vllm_proc = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", VLLM_MODEL,
        "--port", str(VLLM_PORT),
        "--dtype", "auto",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(f"vLLM server starting (PID {vllm_proc.pid}) — waiting 30s for model load...")
time.sleep(30)
print(f"vLLM server should be ready at http://localhost:{VLLM_PORT}")

## Section 3 — Start the Embedding Server

The RAG pipeline embeds query chunks via an HTTP embedding server.
This cell starts a `uvicorn` process serving `embedding_server:app` on
port **8001**.

The `EMBEDDING_SERVER_URL` environment variable (set in Section 4) tells
the pipeline where to POST text for embedding.

In [ ]:
# GPU-only: start the embedding server in the background
import subprocess
import time

EMBED_PORT = 8001

embed_proc = subprocess.Popen(
    [
        "uvicorn", "embedding_server:app",
        "--host", "0.0.0.0",
        "--port", str(EMBED_PORT),
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(f"Embedding server starting (PID {embed_proc.pid}) — waiting 10s...")
time.sleep(10)
print(f"Embedding server should be ready at http://localhost:{EMBED_PORT}")

## Section 4 — Configure Environment Variables

Inject the server URLs and API key into the current process environment.
The ERP RAG application reads these at startup via `os.environ`.

- `EMBEDDING_SERVER_URL` — base URL of the embedding server (Section 3)
- `LLM_SERVER_URL` — base URL of the vLLM server (Section 2)
- `OPENAI_API_KEY` — vLLM does not require a real key; any non-empty string works

This cell is safe to run on CPU-only environments.

In [ ]:
import os

os.environ["EMBEDDING_SERVER_URL"] = "http://localhost:8001"
os.environ["LLM_SERVER_URL"] = "http://localhost:8000"
os.environ["OPENAI_API_KEY"] = "not-required-for-vllm"

# Optional: override the vLLM model name so vLLMLLMClient picks it up
os.environ["VLLM_BASE_URL"] = os.environ["LLM_SERVER_URL"]
os.environ["VLLM_MODEL"] = "mistralai/Mistral-7B-Instruct-v0.2"

print("Environment variables set:")
for key in ("EMBEDDING_SERVER_URL", "LLM_SERVER_URL", "OPENAI_API_KEY", "VLLM_BASE_URL", "VLLM_MODEL"):
    print(f"  {key} = {os.environ[key]}")

## Section 5 — Health Checks

Verify that both servers are reachable before running the RAG pipeline.
A `200 OK` response from each `/health` endpoint confirms the server is
ready to accept requests.

If you see a `ConnectError`, the server process may still be loading the
model — wait 30 seconds and retry.

In [ ]:
import httpx
import os

servers = {
    "LLM (vLLM)": os.environ.get("LLM_SERVER_URL", "http://localhost:8000"),
    "Embedding": os.environ.get("EMBEDDING_SERVER_URL", "http://localhost:8001"),
}

for name, base_url in servers.items():
    health_url = base_url.rstrip("/") + "/health"
    try:
        response = httpx.get(health_url, timeout=5.0)
        status = "OK" if response.status_code == 200 else "UNEXPECTED"
        print(f"[{status}] {name}: {health_url} → {response.status_code}")
    except httpx.ConnectError:
        print(f"[UNREACHABLE] {name}: {health_url} — server not running or still loading")
    except Exception as exc:
        print(f"[ERROR] {name}: {health_url} — {exc}")